In [1]:
import math
import time

print("Các thư viện đã được nạp!")

Các thư viện đã được nạp!


In [2]:
class RoboticArm:
    def __init__(self, home_pos=(0, 0, 0)):
        """Khởi tạo robot tại vị trí 'nhà' (home)."""
        self.current_pos = home_pos
        self.home_pos = home_pos
        self.gripper_engaged = False
        print(f"Cánh tay robot được khởi tạo tại: {self.current_pos}")

    def get_position(self):
        """Báo cáo vị trí hiện tại."""
        return self.current_pos

    def move_to(self, target_pos, speed=10):
        """Mô phỏng việc di chuyển từ vị trí hiện tại đến mục tiêu."""
        start_x, start_y, start_z = self.current_pos
        target_x, target_y, target_z = target_pos

        print(f"\nĐang di chuyển từ {self.current_pos} -> {target_pos}...")

        # Tính toán khoảng cách (distance)
        distance = math.sqrt((target_x - start_x)**2 + (target_y - start_y)**2 + (target_z - start_z)**2)
        steps = int(distance / speed) + 1 # Số bước di chuyển

        for i in range(steps + 1):
            # Nội suy tuyến tính đơn giản
            step_ratio = i / steps
            curr_x = start_x + (target_x - start_x) * step_ratio
            curr_y = start_y + (target_y - start_y) * step_ratio
            curr_z = start_z + (target_z - start_z) * step_ratio

            print(f"  -> Vị trí tạm thời: ({curr_x:.1f}, {curr_y:.1f}, {curr_z:.1f})")
            time.sleep(0.1) # Tạm dừng 0.1s để mô phỏng chuyển động

        # Cập nhật vị trí mới
        self.current_pos = target_pos
        print(f"✅ Đã đến mục tiêu: {self.current_pos}")

    def set_gripper(self, engage: bool):
        """Mô phỏng việc đóng/mở tay gắp."""
        if engage:
            if not self.gripper_engaged:
                print("\n⚡ BỘ GẮP: ĐÃ ĐÓNG (Thu hoạch)")
                self.gripper_engaged = True
            else:
                print("\n⚠️ BỘ GẮP: Đã đóng từ trước.")
        else:
            if self.gripper_engaged:
                print("\n⚡ BỘ GẮP: ĐÃ MỞ (Thả nông sản)")
                self.gripper_engaged = False
            else:
                print("\n⚠️ BỘ GẮP: Đã mở từ trước.")

In [3]:
def convert_bbox_to_3d_coords(bounding_box, depth_cm):
    """
    Mô phỏng việc chuyển đổi tọa độ 2D (pixel) sang 3D (cm).
    bounding_box = [x1, y1, x2, y2]
    depth_cm = Dữ liệu từ cảm biến chiều sâu (ví dụ: Lidar, Camera 3D)
    """
    print(f"\nĐã nhận bounding box: {bounding_box} và độ sâu: {depth_cm} cm")

    # 1. Tìm tâm của bounding box (tính bằng pixel)
    x1, y1, x2, y2 = bounding_box
    center_pixel_x = (x1 + x2) / 2
    center_pixel_y = (y1 + y2) / 2
    print(f"Tâm pixel của mục tiêu: ({center_pixel_x}, {center_pixel_y})")

    # 2. Giả lập phép "Nội suy Camera"
    # Giả sử camera có độ phân giải 1000x1000 và không gian làm việc của robot là 100x100 cm
    CAMERA_RESOLUTION_X = 1000

    # Chuyển đổi pixel X, Y sang cm X, Y
    # Giả sử gốc (0,0) của robot là tâm của camera
    target_robot_x = (center_pixel_x - CAMERA_RESOLUTION_X / 2) / 10
    target_robot_y = (center_pixel_y - CAMERA_RESOLUTION_X / 2) / 10
    target_robot_z = depth_cm # Trục Z chính là độ sâu đo được

    target_3d = (round(target_robot_x, 1), round(target_robot_y, 1), round(target_robot_z, 1))
    print(f"=> Đã tính toán tọa độ 3D của Robot: {target_3d} cm")
    return target_3d

In [4]:
print("======= BẮT ĐẦU CHU TRÌNH THU HOẠCH THÔNG MINH =======")

# 1. Khởi tạo robot tại vị trí 'Home' và vị trí 'Thùng chứa'
home_position = (0, 0, 50)  # Vị trí an toàn (cao 50cm)
bin_position = (-40, 40, 20) # Vị trí thùng chứa
arm = RoboticArm(home_pos=home_position)
time.sleep(1)

# 2. Dữ liệu đầu vào (Giả lập từ 2 notebook trước)
print("--- Robot đang quét tìm mục tiêu... ---")
time.sleep(1)
# Model 1 (ML) trả về: Đã chín
decision_is_ripe = True
# Model 2 (CV) trả về: Vị trí trên ảnh 2D
detected_bounding_box = [600, 450, 650, 500] # Tọa độ pixel [x1, y1, x2, y2]
# Cảm biến độ sâu trả về: Khoảng cách
measured_depth_cm = 35.0

print(f"Model ML: {'Đã chín' if decision_is_ripe else 'Chưa chín'}")
print(f"Model CV: Phát hiện tại {detected_bounding_box}")
print("=======================================================")
time.sleep(2)

# 3. Robot ra quyết định và hành động
if decision_is_ripe:
    print("QUYẾT ĐỊNH: BẮT ĐẦU THU HOẠCH MỤC TIÊU")

    # 3.1. Tính toán tọa độ 3D
    target_coords = convert_bbox_to_3d_coords(detected_bounding_box, measured_depth_cm)
    time.sleep(1)

    # 3.2. Di chuyển đến mục tiêu
    arm.move_to(target_coords)
    time.sleep(1)

    # 3.3. Thực hiện gắp
    arm.set_gripper(engage=True)
    time.sleep(1)

    # 3.4. Quay về vị trí an toàn (Home)
    arm.move_to(home_position)
    time.sleep(1)

    # 3.5. Di chuyển đến thùng chứa
    print("\nĐang di chuyển đến thùng chứa...")
    arm.move_to(bin_position)
    time.sleep(1)

    # 3.6. Thả nông sản
    arm.set_gripper(engage=False)
    time.sleep(1)

    # 3.7. Quay về Home chờ nhiệm vụ tiếp theo
    print("\nQuay về vị trí chờ...")
    arm.move_to(home_position)

    print("\n======= ✅ HOÀN THÀNH CHU TRÌNH THU HOẠCH =======")
else:
    print("QUYẾT ĐỊNH: BỎ QUA MỤC TIÊU (Chưa chín).")
    print("\n======= KẾT THÚC CHU TRÌNH =======")

======= BẮT ĐẦU CHU TRÌNH THU HOẠCH THÔNG MINH =======
Cánh tay robot được khởi tạo tại: (0, 0, 50)
--- Robot đang quét tìm mục tiêu... ---
Model ML: Đã chín
Model CV: Phát hiện tại [600, 450, 650, 500]
QUYẾT ĐỊNH: BẮT ĐẦU THU HOẠCH MỤC TIÊU

Đã nhận bounding box: [600, 450, 650, 500] và độ sâu: 35.0 cm
Tâm pixel của mục tiêu: (625.0, 475.0)
=> Đã tính toán tọa độ 3D của Robot: (12.5, -2.5, 35.0) cm

Đang di chuyển từ (0, 0, 50) -> (12.5, -2.5, 35.0)...
  -> Vị trí tạm thời: (0.0, 0.0, 50.0)
  -> Vị trí tạm thời: (6.2, -1.2, 42.5)
  -> Vị trí tạm thời: (12.5, -2.5, 35.0)
✅ Đã đến mục tiêu: (12.5, -2.5, 35.0)

⚡ BỘ GẮP: ĐÃ ĐÓNG (Thu hoạch)

Đang di chuyển từ (12.5, -2.5, 35.0) -> (0, 0, 50)...
  -> Vị trí tạm thời: (12.5, -2.5, 35.0)
  -> Vị trí tạm thời: (6.2, -1.2, 42.5)
  -> Vị trí tạm thời: (0.0, 0.0, 50.0)
✅ Đã đến mục tiêu: (0, 0, 50)

Đang di chuyển đến thùng chứa...

Đang di chuyển từ (0, 0, 50) -> (-40, 40, 20)...
  -> Vị trí tạm thời: (0.0, 0.0, 50.0)
  -> Vị trí tạm thời: (-5